# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TEJAS7356/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

### Finding 1: The Anatomy of Growing Content

The paper reports that growing content averaged **3,180 words** and was **184 days old**, while declining content averaged **2,311 words** and was **230 days old**.

**Methodology questions:**
- How exactly did the paper define growing and declining content, and over what measurement period?
- Could topic popularity, search demand, or other factors explain the differences rather than content length or age?

The paper presents an observational comparison, so these results do not prove that longer or newer content causes growth.

### Finding 2: AI Model Performance

The paper compares OpenAI and Gemini content cohorts while accounting for content age. The results suggest that their relative performance differs across measurement windows, rather than one model being the clear winner in every case.

**Methodology questions:**
- How was the performance or health score calculated, and does it directly measure content quality?
- Were the content cohorts sufficiently similar in topic, quality, and other factors besides age?

Controlling for age strengthens the comparison, but other differences between the cohorts may still affect the results.

### Suggested Improvements

1. **Growing versus declining content:** Define the performance labels and measurement windows clearly. Test the findings on later time periods and account for differences in topic and search demand.

2. **AI model performance:** Compare OpenAI and Gemini content within matched age and topic groups, using the same metrics and evaluation period. Report sample sizes and uncertainty.

These improvements would strengthen the evidence while keeping the findings observational unless a suitable causal study is conducted.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### 2. My model under an honest split (before/after)

I evaluated my Decision Tree Classifier using two validation strategies.

**Before — Random Split**

I used a random 80/20 train-test split. The model achieved an F1 score of **0.454**. A random split may place content from the same client in both the training and testing sets.

**After — Client-Grouped Split**

I used `GroupShuffleSplit` with `client_hash_id` to ensure that clients in the training set did not appear in the test set. The model achieved an F1 score of **0.478**.

| Validation method | F1 score |
|---|---:|
| Random split (80/20) | 0.454 |
| Client-grouped split (80/20) | 0.478 |
| Difference | +0.023 |

**Interpretation**

The client-grouped split achieved an F1 score approximately 0.023 higher than the random split. This result applies to these particular splits and does not prove that the model will always generalize better to unseen clients.

The target compares raw February and March click totals, although the months have different lengths. Therefore, the results are preliminary. Further validation using comparable measurement windows and additional time periods is needed.

In [14]:
import duckdb
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

feb_monthly = con.sql(f"""
SELECT client_hash_id, content_hash_id,
       SUM(gsc_impressions) AS feb_impressions,
       SUM(gsc_clicks) AS feb_clicks,
       AVG(gsc_avg_position) AS feb_avg_position
FROM {FEB}
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()

mar_monthly = con.sql(f"""
SELECT client_hash_id, content_hash_id,
       SUM(gsc_clicks) AS mar_clicks,
       SUM(gsc_impressions) AS mar_impressions
FROM {MAR}
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()

data = feb_monthly.merge(
    mar_monthly,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

data["target"] = (data["mar_clicks"] < data["feb_clicks"]).astype(int)

print("Matched pairs:", len(data))
print("Dataset ready!")

Matched pairs: 134238
Dataset ready!


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Week 5 model: random vs client-grouped validation
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

features = ["feb_impressions", "feb_clicks", "feb_avg_position"]
X = data[features]
y = data["target"]
groups = data["client_hash_id"]

# Random split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(X_train, y_train)
random_f1 = f1_score(y_test, model.predict(X_test), zero_division=0)

# Client-grouped split
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(X.iloc[train_idx], y.iloc[train_idx])
grouped_f1 = f1_score(
    y.iloc[test_idx], model.predict(X.iloc[test_idx]), zero_division=0
)

print("Random split F1:", round(random_f1, 3))
print("Client-grouped split F1:", round(grouped_f1, 3))
print("Difference:", round(grouped_f1 - random_f1, 3))

Random split F1: 0.446
Client-grouped split F1: 0.478
Difference: 0.031


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

I audited the final feature set for possible data leakage.

1. **Future information leakage:** `mar_clicks` and `mar_impressions` were excluded from the features because they belong to March, the target month. Using them could reveal future information to the model.

2. **Identifier leakage:** `client_hash_id` and `content_hash_id` were excluded from the model features. The client ID was used only to create the grouped train-test split.

3. **Target leakage:** The target was created by comparing March clicks with February clicks. The model used only February impressions, clicks, and average position as input features.

4. **Unequal month lengths:** February has 28 days and March has 31 days. Comparing raw monthly click totals may incorrectly label content as declining. A future version should compare daily click rates or use equivalent time windows.

5. **Selection bias:** The inner join retained only content pairs present in both months. The results may not represent content missing from either month.

6. **Validation leakage:** The random split may place content from the same client in training and testing. The client-grouped split avoids this by keeping each client in only one set.

**Conclusion:** The final feature set avoids direct use of March outcome columns and client identifiers as predictors. However, the target definition and matched-pair selection may still introduce limitations. The F1 scores should be treated as preliminary results, not proof of reliable future predictions.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
features = [
    "feb_impressions",
    "feb_clicks",
    "feb_avg_position"
]

future_columns = ["mar_clicks", "mar_impressions"]
id_columns = ["client_hash_id", "content_hash_id"]

print("Final model features:", features)
print("Future columns excluded:", future_columns)
print("ID columns excluded from features:", id_columns)

assert not any(col in features for col in future_columns)
assert not any(col in features for col in id_columns)

print("Leakage check passed.")

Final model features: ['feb_impressions', 'feb_clicks', 'feb_avg_position']
Future columns excluded: ['mar_clicks', 'mar_impressions']
ID columns excluded from features: ['client_hash_id', 'content_hash_id']
Leakage check passed.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

My Decision Tree Classifier uses February search impressions, clicks, and average position to identify content that may need performance review.

The model achieved an F1 score of 0.425 with a random split and 0.478 with a client-grouped split. The grouped split produced a 0.053 higher F1 score in this evaluation, but this result alone does not prove that the model generalizes better in all cases.

The target compares raw February and March click totals, although the months have different lengths. Natural traffic fluctuations and the use of matched content pairs may also affect the results.

Therefore, the model should be treated as a decision-support tool for prioritizing content for review, not as proof that content is declining or that updating it will improve performance. Further validation with a comparable target and additional time periods is needed.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.